<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/07-instance-segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 实例分割

## 简介

语义分割为每个像素分配一个类别标签，但无法区分同一类别的单个对象。实例分割通过单独检测每个对象并为每个实例生成单独的掩膜来解决此问题。

这种区别对于许多地理空间应用来说至关重要。农业分析师需要单个地块用于作物监测和产量估算，城市规划师需要计算建筑物并测量占地面积，而保护组织则利用树冠描绘来监测森林健康。

随着卫星图像可用性的增长，自动化实例分割变得越来越重要。欧盟的共同农业政策等项目需要数百万个地块的准确地块信息，这使得手动数字化不切实际。

本教程介绍了用于遥感实例分割的 Mask R-CNN。您将下载 [Fields of The World](https://fieldsofthe.world) 基准数据集，准备 Sentinel-2 图像，使用 `geoai` 训练 Mask R-CNN 模型，运行推理，清理和矢量化预测，并提取每个检测到的地块的几何属性。

## 学习目标

在本教程结束时，您将能够：

- 解释实例分割与语义分割有何不同，以及何时适合使用每种方法
- 描述 Mask R-CNN 架构及其关键组件（骨干网络、RPN、检测头、掩膜头）
- 下载并探索用于农田边界检测的 Fields of The World 基准数据集
- 准备 Sentinel-2 多光谱图像和实例分割掩膜用于模型训练
- 使用 `geoai` 包训练用于农田边界描绘的 Mask R-CNN 模型
- 对测试图像运行滑动窗口推理并解释原始预测掩膜
- 通过去除小的虚假检测和填充孔洞来清理实例掩膜
- 将栅格预测矢量化为多边形特征并提取几何属性
- 批量处理多个图像以实现高效的大规模推理

## 实例分割与语义分割

**语义分割**生成单个标签图，其中每个像素属于一个类别。相邻的田地都接收相同的“田地”标签，没有边界，这足以映射总类别范围，但不足以识别单个地块。

**实例分割**为每个检测到的对象生成单独的掩膜，并具有自己的置信度分数和唯一标识符。这使得计数、按对象测量和跨时间跟踪成为可能，但代价是模型复杂性更高。

例如，应用于农业区域的语义分割模型会生成一个单一的连接“农田”区域，无法确定单个田地的数量。实例分割模型会为每个田地生成单独的掩膜，这些掩膜可以转换为具有面积、周长和伸长率等几何属性的多边形。

实例分割和语义分割是互补的。一些架构将两者结合起来进行全景分割，但对于大多数地理空间工作流，特定特征（田地、建筑物、树木）的实例分割是主要用例。

## Mask R-CNN 架构

Mask R-CNN 通过添加掩膜预测分支扩展了 Faster R-CNN 对象检测框架。该架构有四个主要组件。

### 骨干编码器

骨干网络是一个 CNN（通常是 ResNet-50 或 ResNet-101），结合特征金字塔网络 (FPN)，它在多个尺度上提取分层特征图。对于具有四个波段（R、G、B、NIR）的 Sentinel-2 图像，第一个卷积层经过调整以接受四个输入通道。

### 区域提案网络

RPN 扫描特征图并提出可能包含对象的矩形区域。它生成对象性分数和边界框调整，然后通过非最大抑制过滤提案以消除冗余重叠框。

### 检测头

每个提出的区域都使用 RoI Align 从特征图中采样。检测头将每个区域分类为对象类别（或背景）并优化边界框坐标。

### 掩膜头

对于每个检测到的对象，一个小型全卷积网络预测边界框区域内的二进制掩膜。掩膜头通过一个单独的分支操作，因此掩膜预测不会干扰类别或框预测。输出是一个固定大小的掩膜（通常是 28 x 28 像素），调整大小以匹配检测到的边界框。

### RoI Align：保持空间精度

RoI Align 使用双线性插值在精确所需位置计算特征值，无需四舍五入，消除了旧版 RoI Pooling 的量化误差。这种精度对于像农田这样的小对象至关重要，这些对象在训练芯片中可能只占几十个像素。

### 完整流程

骨干网络提取多尺度特征，RPN 提出候选区域，检测头预测类别标签和边界框，掩膜头预测每个实例的二进制掩膜。经过非最大抑制后，输出是一组实例，每个实例都带有一个类别标签、置信度分数、边界框和像素级掩膜。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install -U "geoai-py[extra]"

## 下载 FTW 数据集

[Fields of The World](https://fieldsofthe.world) 数据集是一个用于农田边界检测的大规模基准。它包含来自 24 个国家的 70,462 个图像芯片，将 Sentinel-2 图像（四个波段，10 米分辨率）与实例分割掩膜配对。该数据集可在 [Source Cooperative](https://source.coop/kerner-lab/fields-of-the-world) 上获取。

我们使用 [卢森堡子集](https://source.coop/kerner-lab/fields-of-the-world/luxembourg)，它是最小的国家子集之一，非常适合本教程。

In [ ]:
from pathlib import Path

import geopandas as gpd
import geoai

In [ ]:
geoai.download_ftw(countries=["luxembourg"], output_dir="ftw_data")

### 探索数据集

FTW 数据集包含一个 GeoParquet 文件，其中包含每个芯片的元数据和几何形状，包括官方的训练、验证和测试分割。

In [ ]:
country_dir = Path("ftw_data") / "luxembourg"
chips_gdf = gpd.read_parquet(country_dir / "chips_luxembourg.parquet")

print(f"Total chips: {len(chips_gdf)}")
print(f"\nSplit distribution:")
print(chips_gdf["split"].value_counts())

可视化卢森堡各地训练、验证和测试芯片的空间分布。

In [ ]:
geoai.view_vector_interactive(chips_gdf, column="split")

显示示例图像-掩膜对。每个掩膜使用唯一的整数 ID 来区分单个田地实例。

In [ ]:
geoai.display_ftw_samples("ftw_data", country="luxembourg", num_samples=4)

## 准备训练数据

`geoai` Mask R-CNN 管道需要包含 uint8 GeoTIFF 文件的 `images/` 和 `labels/` 目录。`prepare_ftw` 函数将原始 Sentinel-2 反射率值 (0-10,000) 重新缩放至 0-255，将文件组织成预期结构，并预留测试芯片用于推理。

In [ ]:
data = geoai.prepare_ftw("ftw_data", country="luxembourg")
data

```text
FTW 卢森堡：总计 808 个芯片
  训练：643，验证：81，测试：84
  使用 724 个芯片进行训练
正在准备训练数据...
已准备 724 个训练芯片（跳过 0 个）
已准备 5 个测试芯片
```

该函数将 643 个训练芯片和 81 个验证芯片合并为 724 个芯片用于模型开发。在训练期间，会使用 `val_split=0.2` 创建一个新的验证分割。

通过显示一些图像-标签对来验证准备好的瓦片。

In [ ]:
geoai.display_training_tiles(
    output_dir="field_boundaries",
    num_tiles=4,
    figsize=(12, 6),
    cmap="tab20",
)

## 训练 Mask R-CNN 模型

我们使用 ResNet-50 + FPN 骨干网络训练一个 Mask R-CNN 模型。关键参数：

- **`num_classes=2`**：背景 (0) 和田地 (1)。所有田地都属于一个类别；通过单独的掩膜来区分实例。
- **`num_channels=4`**：Sentinel-2 波段（红、绿、蓝、近红外）。近红外波段有助于区分 RGB 中不可见的植被边界。
- **`instance_labels=True`**：FTW 掩膜已经为每个田地编码了唯一的实例 ID，因此 `geoai` 直接使用它们。
- **`num_epochs=20`**：对于教程来说足够了；生产环境中增加到 50-100 个周期。
- **`val_split=0.2`**：创建内部验证分割，用于在训练期间监控性能。

In [ ]:
geoai.train_instance_segmentation_model(
    images_dir=data["images_dir"],
    labels_dir=data["labels_dir"],
    output_dir="field_boundaries/models",
    num_classes=2,
    num_channels=4,
    batch_size=4,
    num_epochs=20,
    learning_rate=0.005,
    val_split=0.2,
    instance_labels=True,
    visualize=True,
    verbose=True,
)

总损失是 RPN 对象性损失、分类损失、边界框回归损失和掩膜损失的加权和。稳定下降的损失表明健康的训练。如果遇到内存不足错误，请将批量大小减小到 2 或使用更轻的骨干网络。

绘制训练指标以评估收敛性。

In [ ]:
geoai.plot_performance_metrics(
    history_path="field_boundaries/models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

## 运行推理

使用滑动窗口推理将训练好的模型应用于测试图像。128 像素的重叠可防止靠近瓦片边界的对象被分割。通过 `vectorize=True`，该函数返回实例掩膜、类别标签、置信度分数和矢量化多边形。

In [ ]:
test_images = sorted(Path(data["test_dir"]).glob("*.tif"))
test_image_path = str(test_images[0])
masks_path = "field_boundary_prediction.tif"
model_path = "field_boundaries/models/best_model.pth"

result = geoai.instance_segmentation(
    input_path=test_image_path,
    output_path=masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=4,
    window_size=256,
    overlap=128,
    confidence_threshold=0.5,
    batch_size=4,
    vectorize=True,
    class_names=["background", "building"],
)
result

### 可视化原始预测

结果字典包含四个键：`"instance"`（每个字段的唯一整数 ID）、`"class_label"`（预测类别）、`"score"`（置信度）和 `"vector"`（每个字段一个多边形的 GeoDataFrame）。

可视化实例掩膜，其中每种颜色代表一个不同的检测到的字段。

In [ ]:
geoai.view_raster(
    result["instance"],
    nodata=0,
    cmap="tab20",
    basemap=test_image_path,
    backend="ipyleaflet",
)

可视化类别标签栅格和置信度分数栅格。

In [ ]:
geoai.view_raster(
    result["class_label"],
    nodata=0,
    cmap="binary",
    basemap=test_image_path,
    backend="ipyleaflet",
)

In [ ]:
geoai.view_raster(
    result["score"], nodata=0, basemap=test_image_path, backend="ipyleaflet"
)

显示按置信度分数着色的矢量化预测。

In [ ]:
geoai.view_vector_interactive(result["vector"], tiles=test_image_path, column="score")

## 后处理预测

原始实例掩膜通常包含小的虚假检测或孔洞。`clean_instance_mask` 函数会移除小于最小面积的连通分量，并填充指定大小的孔洞。

In [ ]:
cleaned_masks_path = "field_boundary_prediction_cleaned.tif"
geoai.clean_instance_mask(
    result["instance"], cleaned_masks_path, min_area=100, max_hole_area=100
)

In [ ]:
geoai.view_raster(
    cleaned_masks_path,
    nodata=0,
    cmap="tab20",
    basemap=test_image_path,
    backend="ipyleaflet",
)

### 矢量化预测

将清理后的栅格掩膜转换为矢量多边形。每个唯一的像素值都成为地理参考 GeoDataFrame 中的一个独立多边形特征。

In [ ]:
output_vector_path = "field_boundary_prediction.geojson"
gdf = geoai.raster_to_vector(cleaned_masks_path, output_vector_path)

### 将预测与图像进行比较

使用分屏地图直观地比较检测到的农田边界与原始 Sentinel-2 图像。

In [ ]:
geoai.create_split_map(
    left_layer=gdf,
    right_layer=test_image_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_image_path,
)

## 提取几何属性

计算几何属性将原始预测转换为丰富的属性表，用于空间分析。如果使用经纬度坐标，请首先重新投影到投影坐标系。

| 属性       | 描述                                                       |
| ---------- | ---------------------------------------------------------- |
| **面积**   | 字段大小（公顷），对产量估算和补贴计划至关重要         |
| **周长**   | 边界长度（米），可用于围栏成本估算                       |
| **伸长率** | 长轴/短轴比率，区分带状田地和紧凑地块                   |
| **实心度** | 面积/凸包面积比率，衡量边界不规则性                       |
| **范围**   | 面积/边界框面积比率，表示字段的矩形程度                 |

In [ ]:
gdf_props = geoai.add_geometric_properties(gdf, area_unit="ha", length_unit="m")
gdf_props.head()

In [ ]:
gdf_props.describe()

### 按属性可视化字段

按几何属性着色的交互式地图揭示了字段特征的空间模式。

In [ ]:
geoai.view_vector_interactive(gdf_props, column="area_ha", tiles=test_image_path)

伸长率突出显示带状田地与紧凑地块。

In [ ]:
geoai.view_vector_interactive(gdf_props, column="elongation", tiles=test_image_path)

## 批处理

批处理将相同的模型应用于目录中的所有图像，避免重复加载模型并更好地利用 GPU 资源。

In [ ]:
geoai.instance_segmentation_batch(
    input_dir=data["test_dir"],
    output_dir="field_boundaries/predictions",
    model_path=model_path,
    num_classes=2,
    num_channels=4,
    window_size=256,
    overlap=128,
    confidence_threshold=0.5,
    batch_size=4,
)

## 主要收获

1. 实例分割通过独立的掩膜检测单个对象，与语义分割不同，它能够进行计数和按对象测量。

2. Mask R-CNN 结合了骨干编码器、区域提案网络、检测头和掩膜头，用于分类、定位和分割每个对象实例。

3. Fields of The World 数据集将 Sentinel-2 图像与 24 个国家的实例掩膜配对，以实现可重现的字段边界检测。

4. 近红外波段捕捉 RGB 中不可见的植被差异，改善了相邻字段之间的边界检测。

5. `instance_labels=True` 标志告诉管道直接使用掩膜中预先存在的唯一实例 ID。

6. 使用 `clean_instance_mask` 进行后处理可移除小的虚假检测并填充孔洞，以获得更清晰的输出。

7. `raster_to_vector` 函数将每个唯一的实例 ID 转换为一个独立的 GIS 就绪多边形特征。

8. 几何属性（面积、周长、伸长率、实心度、范围）使研究区域的字段特征能够进行统计分析。

9. 批处理推理通过一次调用处理整个图像目录，避免重复加载模型。